In [6]:
%load_ext autoreload
%autoreload2

UsageError: Line magic function `%autoreload2` not found.


In [1]:
import base64
import pymupdf
from openai import OpenAI
from pricing import calculate_cost_response
from page_extract import Page, PageResponse

openai_client = OpenAI()

doc = pymupdf.open(
    r"/Users/mikifoster/Documents/Code_Projects/rags_to_agents/agent_experimental_builds/input_docs/books/math-engineers.pdf"
)

In [ ]:
def to_base64_url(page):
    matrix = pymupdf.Matrix(1.5, 1.5)
    pixmap = page.get_pixmap(matrix=matrix)
    png_bytes = pixmap.tobytes(output="png")

    b64 = base64.b64encode(png_bytes).decode("utf-8")
    image_url = f"data:image/png;base64,{b64}"
    return image_url

In [3]:
instructions = """
You are extracting a textbook page into structured page blocks.

text and formulas should be extracted verbatim.

use latex for all math.
Use "$" for inline equation and EquationBlock type for block equations.

some inline equations should be treated as block equations
if there's little text around them.

important: don't skip any text. if something is not possible to
recognize, include a placeholder

Extraction rules:
1) Preserve reading order. The blocks list must match the order a human reads the page.
2) Do NOT include OCR or layout details (no coordinates, fonts, line breaks, or scan artifacts).
3) Prefer fewer, larger TextBlocks over many tiny ones. Group adjacent paragraphs when they belong together.
4) Use LaTeX for all math in EquationBlock.latex.
5) Section headings must be SectionHeadingBlock only; do not include body text in them.
6) FigureBlock.description should explain what the figure conveys conceptually (graphs, curves, relationships),
   not how it looks on the page.
7) TableBlock should capture semantic columns and rows. Include units in column names if shown.
8) Store the running page header (if any) in Page.header.
9) If uncertain, make a best-faith concise extraction; do not invent content.
""".strip()


def extract_page_information(
    page, detail="low", model_name="gpt-4o-mini"
) -> PageResponse:
    image_url = to_base64_url(page)

    image_content = {"type": "input_image", "image_url": image_url, "detail": detail}

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": [image_content]},
    ]

    response = openai_client.responses.parse(
        model=model_name, input=messages, text_format=Page
    )

    cost = calculate_cost_response(response)

    return PageResponse(page=response.output_parsed, cost=cost)

In [4]:
page_response = extract_page_information(doc[100])
print(page_response.cost)
page_response.page.print()

0.00077805
83
ADDITIONAL RULES OF DIFFERENTIATION
If $z$ is a function of $x$ and $y$, i.e., $z = f(x, y)$, the total differential $dz$ is obtained from the partial differentials $dx$ and $dy$ by the use of the following rule:

$$dz = \frac{\partial z}{\partial x} dx + \frac{\partial z}{\partial y} dy$$

The reason for this is more clearly seen if we work from the fundamental idea of a curve, and introduce the actually measurable quantities like $x$ and $y$.

Figure 21. Change in position on the surface.
The figure illustrates a surface with a change in position on it, showing the relationship between the changes in $x$ and $y$ and the corresponding change in $z$ as measured by the partial derivatives.
Fig. 21

Thus:

$$\Delta z = \Delta x \cdot \frac{\partial z}{\partial x} + \Delta y \cdot \frac{\partial z}{\partial y}$$

Let P be a point $(x, y)$ on a surface, and let P move to a new position Q near to P. The change of position is limited by reference to a diagram (Fig. 21).



In [5]:
from pathlib import Path
from tqdm.auto import tqdm

output = Path("output/")

In [6]:
for page_number, page in enumerate(tqdm(doc)):
    page_file = output / f"page_{page_number:03d}.json"

    if page_file.exists():
        continue

    page_response = extract_page_information(page)

    page_json = page_response.model_dump_json(indent=2)
    page_file.write_text(page_json, encoding="utf-8")

  0%|          | 0/442 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [7]:
def process_document(page_number) -> bool:
    try:
        page_file = output / f"page_{page_number:03d}.json"

        if page_file.exists():
            print(f"{page_file} already processed")
            return True

        page = doc[page_number]
        page_response = extract_page_information(page)

        page_json = page_response.model_dump_json(indent=2)
        page_file.write_text(page_json, encoding="utf-8")

        return True
    except Exception as e:
        print(e)
        return False

In [8]:
process_document(110)

True

In [10]:
from concurrent.futures import ThreadPoolExecutor


def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

        return results

In [ ]:
with ThreadPoolExecutor(max_workers=4) as pool:
    pages_seq = list(range(len(doc)))
    map_progress(pool, pages_seq, process_document)

  0%|          | 0/442 [00:00<?, ?it/s]

output/page_000.json already processedoutput/page_001.json already processed
output/page_002.json already processed

output/page_005.json already processed
output/page_004.json already processed
output/page_003.json already processed
output/page_006.json already processed
output/page_007.json already processed
output/page_008.json already processed
output/page_011.json already processed
output/page_012.json already processed
output/page_009.json already processed
output/page_010.json already processed
output/page_013.json already processed
output/page_014.json already processed


In [9]:
import requests
import tarfile
import io
import json
from page_extract import PageResponse

url = "https://github.com/alexeygrigorev/ai-engineering-buildcamp-code/releases/download/math-book-pages/math-book-pages.tar.gz"

tag_gz_content = requests.get(url).content
archive = tarfile.open(fileobj=io.BytesIO(tag_gz_content), mode="r:gz")

In [10]:
filename = "output/page_092.json"
file = archive.extractfile(filename)
content = file.read()
book_page = json.loads(content)
page_response = PageResponse.model_validate_json(content)
page_response

PageResponse(page=Page(page_number=75, header='ADDITIONAL RULES OF DIFFERENTIATION', blocks=[TextBlock(type='text', text='Example 12—If $ y = g(t) = 4x + 4x^{2} \\cdot 1929  \\cdot 2 + \\ldots \n\nLet $ u = \\frac{ du}{dx} = 4 + 8x $.'), EquationBlock(type='equation', latex='\\frac{dy}{dx} = \\frac{dy}{du} \\cdot \\frac{du}{dx} = 4 \\cdot (4 + 8x) = \\ldots \n\nHence:\n\n$ \\frac{dy}{dx} = (4 \\cdot 1928 + 8x \\cdot 1928) e^{4y}$.', description='This equation relates the derivative of y with respect to x through the chain rule.'), TextBlock(type='text', text='Example 13—For a spring loaded governor (see Fig. 19).   \n$ W = T + Q(V - \\sqrt{R}) .$'), FigureBlock(type='figure', caption='Fig. 19—Spring loaded Governor.', description='This figure illustrates a spring loaded governor, displaying the relationship between weight (W), tension (T), and other variables.', figure_number=19), TextBlock(type='text', text='Where $ W $ = force to elang the spring in unit, $ T $ = tension in spring, $

In [11]:
def block_to_string(block):
    lines = []

    if block.type == "text":
        lines.append(block.text)

    elif block.type == "equation":
        lines.append(f"$${block.latex}$$")

    elif block.type == "figure":
        lines.append(block.caption or "")
        lines.append(block.description or "")
        lines.append(f"Fig. {block.figure_number}")

    else:
        lines.append(str(block))

    return "\n".join(lines)


def blocks_to_string(blocks):
    lines = []

    for block in blocks:
        lines.append(block_to_string(block))

    return "\n".join(lines)

In [12]:
documents = []

for file_info in archive.getmembers():
    filename = file_info.name

    if not filename.endswith(".json"):
        continue

    file = archive.extractfile(filename)
    content = file.read()
    page_response = PageResponse.model_validate_json(content)

    page = page_response.page

    content = blocks_to_string(page.blocks)

    doc = {
        "filename": filename,
        "content": content,
    }

    documents.append(doc)

archive.close()

In [14]:
archive.close()

In [17]:
from minsearch import Index
from openai import OpenAI
import rag

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(documents)

openai_client = OpenAI()

book_rag = rag.RAG(
    index=index,
    llm_client=openai_client,
)

response = book_rag.rag("double sum curve")

print(response.answer)

The "Double Sum Curve" method relates to finding the centroid vertical of an area under the sum curve denoted as PQ. It involves using a graphical approach to derive the centroid's location based on the areas represented by the corresponding curves. A diagram typically illustrates this method, showing how the centroid vertical can be created from the curve PQ and involves parallel distances to the origin of the curve.


In [18]:
import io
import tarfile

import requests

from page_extract import PageResponse

url = 'https://github.com/alexeygrigorev/ai-engineering-buildcamp-code/releases/download/math-book-pages/math-book-pages.tar.gz'

tag_gz_content = requests.get(url).content

archive = tarfile.open(fileobj=io.BytesIO(tag_gz_content), mode="r:gz")


def block_to_string(block):
    lines = []

    if block.type == "text":
        lines.append(block.text)

    elif block.type == "equation":
        lines.append(f"$${block.latex}$$")

    elif block.type == "figure":
        lines.append(block.caption or "")
        lines.append(block.description or "")
        lines.append(f"Fig. {block.figure_number}")

    else:
        lines.append(str(block))

    return "\n".join(lines)


def blocks_to_string(blocks):
    lines = []

    for block in blocks:
        lines.append(block_to_string(block))

    return "\n".join(lines)

In [19]:
documents = []

for file_info in archive.getmembers():
    filename = file_info.name

    if not filename.endswith(".json"):
        continue

    file = archive.extractfile(filename)
    content = file.read()
    page_response = PageResponse.model_validate_json(content)

    page = page_response.page

    content = blocks_to_string(page.blocks)

    doc = {
        "filename": filename,
        "content": content,
    }

    documents.append(doc)

In [21]:
archive.close()
from pathlib import Path
import time
from sqlitesearch import TextSearchIndex

Path("db").mkdir(parents=True, exist_ok=True)


index = TextSearchIndex(text_fields=["content"], db_path="db/math_book_2.sqlite")

In [22]:
for doc in documents:
    print(doc["filename"])
    index.add(doc)
    time.sleep(0.5)

output/page_000.json
output/page_001.json
output/page_002.json
output/page_003.json
output/page_004.json
output/page_005.json
output/page_006.json
output/page_007.json
output/page_008.json
output/page_009.json
output/page_010.json
output/page_011.json
output/page_012.json
output/page_013.json
output/page_014.json
output/page_015.json
output/page_016.json
output/page_017.json
output/page_018.json
output/page_019.json
output/page_020.json
output/page_021.json
output/page_022.json
output/page_023.json
output/page_024.json
output/page_025.json
output/page_026.json
output/page_027.json
output/page_028.json
output/page_029.json
output/page_030.json
output/page_031.json
output/page_032.json
output/page_033.json
output/page_034.json
output/page_035.json
output/page_036.json
output/page_037.json
output/page_038.json
output/page_039.json
output/page_040.json
output/page_041.json
output/page_042.json
output/page_043.json
output/page_044.json
output/page_045.json
output/page_046.json
output/page_0